# Huấn luyện MIND-large — biến thể 15 field không body

Notebook này chạy `mind_news_metadata.full_pipeline` trên Kaggle. Mặc định là **fresh từ update 0**, partition 0, effective batch 1024 và microbatch lấy từ cấu hình chuẩn. Training tự lưu tối đa hai checkpoint gần nhất theo cấu hình dự án. Notebook không chạy preprocessing/GloVe, không nạp checkpoint cũ ở chế độ fresh và không upload dữ liệu ra dịch vụ ngoài Kaggle.

Cần đính kèm hai input dataset ở chế độ Private:

- `drakan02/mind-15-field-training-inputs`
- Runtime source Private: mã nguồn và `MIND_RUNTIME_SOURCE_MANIFEST.json` có thể nằm ở hai input dataset riêng. Cũng có thể dùng ZIP `mind_15field_runtime_source.zip`; notebook kiểm tra payload SHA-256 trước khi nạp code.

Chọn GPU. Cell môi trường tạo Python 3.11 + TensorFlow 2.15.1 trong virtualenv riêng, không thay TensorFlow của kernel Kaggle. Cần bật Internet cho bước cài gói nếu virtualenv chưa tồn tại. Notebook giới hạn TensorFlow dùng GPU 0, không bật multi-GPU hay mixed precision.

Một epoch partition 0 có thể dài hơn thời lượng một phiên Kaggle. Nếu phiên bị ngắt, chỉ checkpoint đã ghi hoàn chỉnh mới dùng làm mốc. Lưu phiên Notebook Private để giữ output; không coi run bị ngắt là một epoch hoàn tất.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time

# Input dữ liệu giữ slug đã xác nhận. Source runtime có thể được gắn dưới dạng
# cây đã giải nén hoặc manifest/ZIP ở các dataset riêng; chỉ nhận đúng payload.
KAGGLE_DATASETS_ROOT = Path("/kaggle/input/datasets/drakan02")
INPUT_DATASET_DIR = KAGGLE_DATASETS_ROOT / "mind-15-field-training-inputs"
RUNTIME_MANIFEST_NAME = "MIND_RUNTIME_SOURCE_MANIFEST.json"
EXPECTED_RUNTIME_PAYLOAD_SHA256 = "e47a215bfb362cc5760204e61111da07811296663040653a553edd2fdfb151c9"
source_dataset_roots = sorted(
    p for p in KAGGLE_DATASETS_ROOT.iterdir()
    if p.is_dir() and (p / RUNTIME_MANIFEST_NAME).is_file()
)
if not source_dataset_roots:
    raise FileNotFoundError(
        "Không tìm thấy input runtime có " + RUNTIME_MANIFEST_NAME
        + " dưới " + str(KAGGLE_DATASETS_ROOT)
    )
source_manifests = {}
for root in source_dataset_roots:
    manifest_path = root / RUNTIME_MANIFEST_NAME
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        payload = manifest.get("payload_sha256")
    except Exception as exc:
        raise ValueError("Không đọc được runtime manifest: " + str(manifest_path) + ": " + repr(exc))
    source_manifests[str(root)] = payload
matching_source_roots = [
    root for root in source_dataset_roots
    if source_manifests[str(root)] == EXPECTED_RUNTIME_PAYLOAD_SHA256
]
if not matching_source_roots:
    raise ValueError(
        "Không có runtime source khớp payload code đã sửa. Expected="
        + EXPECTED_RUNTIME_PAYLOAD_SHA256
        + "; manifests tìm thấy=" + json.dumps(source_manifests, ensure_ascii=False)
        + ". Gắn manifest và source package mới; không chạy code có checksum khác."
    )
# Có thể có manifest cùng payload trong cây source và một dataset manifest riêng.
# Giữ tất cả root tương ứng; cell kế tiếp chọn cây đầy đủ hoặc archive đã xác minh.
PIPELINE_DATASET_DIR = matching_source_roots[0]

# Fresh là mặc định và bắt đầu update 0. Resume chỉ chạy khi người dùng chủ động
# đổi RUN_MODE và gắn một output run trước làm input; không tự dò hay fallback.
RUN_MODE = "fresh"  # "fresh" hoặc "resume"
RESUME_RUN_DIR = ""  # đường dẫn run đã lưu, chỉ dùng khi RUN_MODE="resume"
PARTITION = 0

WORK = Path("/kaggle/working")
RUNS_ROOT = WORK / "recommendation_runs"
MIN_FREE_DISK_BYTES = 5 * 1024**3
TF_ENV = WORK / "mind-tf215-gpu"
TF_PYTHON = TF_ENV / "bin/python"

assert RUN_MODE in {"fresh", "resume"}
assert INPUT_DATASET_DIR.is_dir(), f"Không tìm thấy input dataset: {INPUT_DATASET_DIR}"
assert PIPELINE_DATASET_DIR.is_dir(), f"Không tìm thấy source dataset: {PIPELINE_DATASET_DIR}"
assert RUN_MODE != "resume" or RESUME_RUN_DIR, "Resume cần RESUME_RUN_DIR được chỉ rõ; không tự chọn checkpoint."
print(json.dumps({
    "python_kernel": sys.version.split()[0],
    "platform": platform.platform(),
    "input_dataset_dir": str(INPUT_DATASET_DIR),
    "pipeline_dataset_dir": str(PIPELINE_DATASET_DIR),
    "run_mode": RUN_MODE,
    "partition": PARTITION,
}, indent=2))

In [ ]:
# Kiểm tra GPU host. Đây chưa phải kiểm tra TensorFlow; phép toán thật sẽ được
# kiểm tra trong virtualenv trước khi training.
sm = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.free",
     "--format=csv,noheader"],
    capture_output=True, text=True)
print(sm.stdout)
if sm.returncode != 0 or not sm.stdout.strip():
    raise RuntimeError("Kaggle runtime không cung cấp GPU hoạt động; dừng trước khi cài/training.\n" + sm.stderr)
print("GPU host thấy được:", len(sm.stdout.strip().splitlines()))
print("Training dùng duy nhất GPU 0 để theo profile một GPU của báo cáo.")

In [ ]:
# Tìm đúng source root và artifact root theo cấu trúc file, không giả định manifest
# cũ có đường dẫn tương đối hoàn hảo. Không đọc nội dung SQLite/matrix ở cell này.
REQUIRED_CODE_FILES = [
    "mind_config.py",
    "configs/mind_no_body_15field.json",
    "mind_data_pipeline/__init__.py",
    "mind_data_pipeline/config.py",
    "mind_data_pipeline/data.py",
    "mind_data_pipeline/preprocess.py",
    "mind_news_metadata/__init__.py",
    "mind_news_metadata/batch_data.py",
    "mind_news_metadata/encoder_config.py",
    "mind_news_metadata/field_encoder.py",
    "mind_news_metadata/recommendation_model.py",
    "mind_news_metadata/recommendation_protocol.py",
    "mind_news_metadata/full_pipeline.py",
    "mind_news_metadata/prediction.py",
    "mind_news_metadata/build_train_did_membership.py",
]

def unique_paths(paths):
    result, seen = [], set()
    for p in paths:
        p = Path(p)
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            result.append(p)
    return result

# Source files and their manifest may be attached as separate Kaggle datasets.
# Verify candidate trees against the selected manifest payload before use.
all_input_roots = sorted(p for p in KAGGLE_DATASETS_ROOT.iterdir() if p.is_dir())
code_roots = list(all_input_roots)
code_roots += [p.parent for root in all_input_roots for p in root.rglob("mind_config.py")]
code_root = None
code_missing = None
invalid_source_trees = []
expected_commit = "f632bdf7b1136dff142835f9fe69c8ce5f8c166d"
for candidate in unique_paths(code_roots):
    missing = [rel for rel in REQUIRED_CODE_FILES if not (candidate / rel).is_file()]
    if missing:
        if code_missing is None or len(missing) < len(code_missing):
            code_missing = missing
        continue
    local_manifest_path = candidate / RUNTIME_MANIFEST_NAME
    manifest_path = (local_manifest_path if local_manifest_path.is_file()
                     else matching_source_roots[0] / RUNTIME_MANIFEST_NAME)
    try:
        package_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if package_manifest.get("upstream_commit") != expected_commit:
            raise ValueError("upstream commit mismatch")
        if package_manifest.get("payload_sha256") != EXPECTED_RUNTIME_PAYLOAD_SHA256:
            raise ValueError("payload hash mismatch")
        actual_hashes = {}
        for row in package_manifest.get("files", []):
            rel = Path(row["path"])
            if rel.is_absolute() or ".." in rel.parts:
                raise ValueError("unsafe source path: " + str(rel))
            src = candidate / rel
            if not src.is_file() or src.stat().st_size != int(row["bytes"]):
                raise ValueError("source file missing/size mismatch: " + str(rel))
            digest = hashlib.sha256(src.read_bytes()).hexdigest()
            if digest != row["sha256"]:
                raise ValueError("source checksum mismatch: " + str(rel))
            actual_hashes[rel.as_posix()] = digest
        actual_payload = hashlib.sha256(
            json.dumps(actual_hashes, sort_keys=True).encode("utf-8")
        ).hexdigest()
        if actual_payload != EXPECTED_RUNTIME_PAYLOAD_SHA256:
            raise ValueError("verified source tree payload mismatch")
        code_root = candidate
        break
    except Exception as exc:
        invalid_source_trees.append({"root": str(candidate), "error": repr(exc)})

if code_root is None:
    # The ZIP can be in a different attached Kaggle dataset from the manifest.
    # Ignore unrelated archives and require the exact locked payload.
    import zipfile
    from pathlib import PurePosixPath

    archives = sorted(KAGGLE_DATASETS_ROOT.rglob("mind_15field_runtime_source.zip"))
    matching_archives = []
    archive_errors = []
    for archive_path in archives:
        try:
            with zipfile.ZipFile(archive_path, "r") as archive:
                names = archive.namelist()
                if len(names) != len(set(names)):
                    raise ValueError("duplicate archive member names")
                manifest_name = RUNTIME_MANIFEST_NAME
                if manifest_name not in names:
                    raise ValueError("archive manifest missing")
                package_manifest = json.loads(archive.read(manifest_name).decode("utf-8"))
                if package_manifest.get("payload_sha256") == EXPECTED_RUNTIME_PAYLOAD_SHA256:
                    matching_archives.append(archive_path)
        except Exception as exc:
            archive_errors.append({"archive": str(archive_path), "error": repr(exc)})
    if len(matching_archives) != 1:
        raise FileNotFoundError(
            "Không tìm được đúng một source runtime đã sửa và xác minh. "
            "Cần cây source đầy đủ hoặc mind_15field_runtime_source.zip có payload "
            + EXPECTED_RUNTIME_PAYLOAD_SHA256
            + "; matching_archives=" + json.dumps([str(x) for x in matching_archives])
            + "; invalid_source_trees=" + json.dumps(invalid_source_trees, ensure_ascii=False)
            + "; archive_errors=" + json.dumps(archive_errors, ensure_ascii=False)
            + ". Không dùng code có manifest/checksum khác."
        )

    archive_path = matching_archives[0]
    with zipfile.ZipFile(archive_path, "r") as archive:
        names = archive.namelist()
        package_manifest = json.loads(archive.read(RUNTIME_MANIFEST_NAME).decode("utf-8"))
        if package_manifest.get("upstream_commit") != expected_commit:
            raise ValueError("Runtime archive upstream commit không khớp source lock.")
        rows = package_manifest.get("files", [])
        code_hashes = {}
        code_bytes = {}
        for row in rows:
            member = row["path"]
            rel = PurePosixPath(member)
            if rel.is_absolute() or ".." in rel.parts or "\\" in member:
                raise ValueError("Unsafe path in runtime source archive: " + member)
            if member not in names:
                raise ValueError("Runtime archive thiếu member: " + member)
            payload = archive.read(member)
            if len(payload) != int(row["bytes"]):
                raise ValueError("Runtime source byte count mismatch: " + member)
            digest = hashlib.sha256(payload).hexdigest()
            if digest != row["sha256"]:
                raise ValueError("Runtime source checksum mismatch: " + member)
            code_hashes[member] = digest
            code_bytes[member] = payload
        actual_payload_hash = hashlib.sha256(
            json.dumps(code_hashes, sort_keys=True).encode("utf-8")
        ).hexdigest()
        if actual_payload_hash != package_manifest.get("payload_sha256"):
            raise ValueError("Runtime source payload fingerprint mismatch.")
        if actual_payload_hash != EXPECTED_RUNTIME_PAYLOAD_SHA256:
            raise ValueError("Runtime archive is not the expected fixed source payload.")
        expected_members = set(code_bytes) | {RUNTIME_MANIFEST_NAME}
        if set(names) != expected_members:
            raise ValueError("Runtime source archive chứa member ngoài manifest.")

    extracted_root = WORK / ("mind15_runtime_source_" + actual_payload_hash[:12])
    extracted_root.mkdir(parents=True, exist_ok=True)
    for member, payload in code_bytes.items():
        target = extracted_root / Path(*PurePosixPath(member).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != code_hashes[member]:
            raise FileExistsError("Extracted source file exists with different bytes: " + str(target))
        if not target.exists():
            target.write_bytes(payload)
    manifest_target = extracted_root / RUNTIME_MANIFEST_NAME
    manifest_payload = json.dumps(package_manifest, indent=2, sort_keys=True) + "\n"
    if manifest_target.exists() and manifest_target.read_text(encoding="utf-8") != manifest_payload:
        raise FileExistsError("Extracted source manifest differs; refusing overwrite.")
    if not manifest_target.exists():
        manifest_target.write_text(manifest_payload, encoding="utf-8")
    missing = [rel for rel in REQUIRED_CODE_FILES if not (extracted_root / rel).is_file()]
    if missing:
        raise FileNotFoundError("Verified archive lacks required runtime files: " + json.dumps(missing))
    code_root = extracted_root
    print("Runtime source loaded from verified archive:", archive_path)

# Verify source tree bytes against its source-only manifest before runtime copy.
source_manifest_path = code_root / RUNTIME_MANIFEST_NAME
if not source_manifest_path.is_file():
    # The source tree and its matching manifest may be separate Kaggle inputs.
    source_manifest_path = matching_source_roots[0] / RUNTIME_MANIFEST_NAME
source_package_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
expected_commit = "f632bdf7b1136dff142835f9fe69c8ce5f8c166d"
if source_package_manifest.get("upstream_commit") != expected_commit:
    raise ValueError("Source package upstream commit không khớp source lock.")
if source_package_manifest.get("payload_sha256") != EXPECTED_RUNTIME_PAYLOAD_SHA256:
    raise ValueError("Source package payload is not the fixed corrected runtime bundle.")
manifest_hashes = {}
for row in source_package_manifest.get("files", []):
    rel = Path(row["path"])
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError("Unsafe path in source manifest: " + str(rel))
    path = code_root / rel
    if not path.is_file() or path.stat().st_size != int(row["bytes"]):
        raise ValueError("Source package file missing/size mismatch: " + str(rel))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != row["sha256"]:
        raise ValueError("Source package checksum mismatch: " + str(rel))
    manifest_hashes[rel.as_posix()] = digest
manifest_payload_hash = hashlib.sha256(
    json.dumps(manifest_hashes, sort_keys=True).encode("utf-8")
).hexdigest()
if manifest_payload_hash != source_package_manifest.get("payload_sha256"):
    raise ValueError("Source package payload checksum mismatch.")
if any(not (code_root / rel).is_file() for rel in REQUIRED_CODE_FILES):
    raise FileNotFoundError("Verified source manifest does not cover all required training files.")

REQUIRED_ARTIFACTS = [
    "artifact_manifest.json",
    "embeddings/word_glove_title_abstract_l2.npy",
    "embeddings/word_glove_title_abstract_l2.manifest.json",
    "embeddings/word_glove_title_abstract.npy",
    "embeddings/word_glove_title_abstract.manifest.json",
    "embeddings/entity.npy",
    "embeddings/entity_matrix_manifest.json",
    "vocab/word.tsv",
    "vocab/entity.tsv",
]
artifact_roots = [INPUT_DATASET_DIR, INPUT_DATASET_DIR / "artifacts" / "mind_large_prepared"]
artifact_roots += [p.parent for p in INPUT_DATASET_DIR.rglob("artifact_manifest.json")]
prepared_root = None
artifact_missing = None
for candidate in unique_paths(artifact_roots):
    missing = [rel for rel in REQUIRED_ARTIFACTS if not (candidate / rel).is_file()]
    if not missing:
        prepared_root = candidate
        break
    if artifact_missing is None or len(missing) < len(artifact_missing):
        artifact_missing = missing
if prepared_root is None:
    raise FileNotFoundError(
        "Training input thiếu artifacts ngoài DB: "
        + json.dumps(artifact_missing, ensure_ascii=False)
        + ". Không dùng checkpoint/vocab thay thế và không chạy preprocessing ở Kaggle."
    )

# DB có thể được đóng gói ở một thư mục con khác trong cùng input dataset.
# Chỉ chấp nhận đúng stage01.sqlite được khai báo trong artifact manifest.
artifact_manifest_path = prepared_root / "artifact_manifest.json"
artifact_manifest = json.loads(artifact_manifest_path.read_text(encoding="utf-8"))
artifact_entries = {row.get("path"): row for row in artifact_manifest.get("artifacts", [])}
db_entry = artifact_entries.get("stage01.sqlite")
if not isinstance(db_entry, dict) or int(db_entry.get("bytes", 0)) <= 0:
    raise ValueError("artifact_manifest.json không khóa stage01.sqlite với byte count hợp lệ.")
expected_db_bytes = int(db_entry["bytes"])

# Không đọc/hash toàn bộ DB ở bước dò. Kaggle input là read-only; kích thước
# phải khớp manifest, còn provenance/checksum đã khóa trong manifest transfer.
db_candidates = unique_paths(
    [p for p in [prepared_root / "stage01.sqlite"] if p.is_file()]
    + [p for p in INPUT_DATASET_DIR.rglob("stage01.sqlite") if p.is_file()]
)
matching_db_candidates = [p for p in db_candidates if p.stat().st_size == expected_db_bytes]
if len(matching_db_candidates) != 1:
    diagnostics = [
        {"path": str(p), "bytes": p.stat().st_size, "matches_manifest_bytes": p.stat().st_size == expected_db_bytes}
        for p in db_candidates
    ]
    if not db_candidates:
        detail = "Không tìm thấy file nào tên stage01.sqlite trong input dataset."
    elif not matching_db_candidates:
        detail = "Có stage01.sqlite nhưng không file nào khớp byte count trong manifest."
    else:
        detail = "Có nhiều stage01.sqlite khớp byte count; từ chối chọn mơ hồ."
    raise FileNotFoundError(
        "Training input thiếu hoặc không xác định được đúng DB: " + detail
        + " expected_bytes=" + str(expected_db_bytes)
        + " candidates=" + json.dumps(diagnostics, ensure_ascii=False)
        + ". Cập nhật đúng phiên bản Private input dataset; không thay DB và không chạy preprocessing."
    )
DB_SOURCE_PATH = matching_db_candidates[0]

config_data = json.loads((code_root / "configs/mind_no_body_15field.json").read_text(encoding="utf-8"))
expected_commit = "f632bdf7b1136dff142835f9fe69c8ce5f8c166d"
if config_data.get("source", {}).get("commit") != expected_commit:
    raise ValueError("Source config không trỏ tới commit đã khóa: " + str(config_data.get("source", {}).get("commit")))
if config_data.get("variant") != "mind_no_body_15field":
    raise ValueError("Sai variant cấu hình: " + str(config_data.get("variant")))
if len(config_data["encoder"]["field_order"]) != 15:
    raise ValueError("Notebook chỉ chấp nhận hợp đồng 15 field không body")

# Tạo runtime copy nhỏ chỉ từ Python/JSON trong các package; input artifact lớn
# được symlink read-only, không copy DB, vocabulary hoặc ma trận.
source_files = []
for top in ("mind_data_pipeline", "mind_news_metadata", "configs"):
    for path in sorted((code_root / top).rglob("*")):
        if path.is_file() and path.suffix in {".py", ".json"}:
            source_files.append(path)
source_files.append(code_root / "mind_config.py")
for optional in ("source_lock.json", "source_manifest.json", "source_provenance.json"):
    if (code_root / optional).is_file():
        source_files.append(code_root / optional)
source_files = unique_paths(source_files)
source_hashes = {
    str(path.relative_to(code_root)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in source_files
}
payload_hash = hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest()
PROJECT_ROOT = WORK / ("mind15_project_runtime_v2_" + payload_hash[:12])
if PROJECT_ROOT.exists():
    saved = PROJECT_ROOT / "KAGGLE_RUNTIME_SOURCE_MANIFEST.json"
    if not saved.is_file() or json.loads(saved.read_text()).get("payload_sha256") != payload_hash:
        raise FileExistsError("Runtime source dir đã tồn tại nhưng không khớp; không ghi đè: " + str(PROJECT_ROOT))
else:
    for source in source_files:
        target = PROJECT_ROOT / source.relative_to(code_root)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    (PROJECT_ROOT / "KAGGLE_RUNTIME_SOURCE_MANIFEST.json").write_text(json.dumps({
        "source_dataset_dir": str(PIPELINE_DATASET_DIR),
        "source_root": str(code_root),
        "upstream_commit": expected_commit,
        "payload_sha256": payload_hash,
        "files": [{"path": k, "sha256": v} for k, v in sorted(source_hashes.items())],
    }, indent=2) + "\n")

ARTIFACT_LINK = PROJECT_ROOT / "artifacts" / "mind_large_prepared"
ARTIFACT_LINK.mkdir(parents=True, exist_ok=True)
# Overlay chỉ gồm đúng các artifact đã khóa. DB có thể nằm lồng trong input;
# symlink giúp pipeline dùng path chuẩn mà không copy hay sửa dữ liệu.
artifact_sources = {
    "artifact_manifest.json": artifact_manifest_path,
    "stage01.sqlite": DB_SOURCE_PATH,
    "embeddings": prepared_root / "embeddings",
    "vocab": prepared_root / "vocab",
}
for name, target in artifact_sources.items():
    link = ARTIFACT_LINK / name
    target = target.resolve()
    if link.is_symlink():
        if link.resolve() != target:
            raise RuntimeError("Artifact symlink đã trỏ sang nguồn khác; dừng, không ghi đè: " + str(link))
    elif link.exists():
        raise RuntimeError("Artifact path đã tồn tại và không phải symlink; dừng: " + str(link))
    else:
        link.symlink_to(target, target_is_directory=target.is_dir())

DB_PATH = ARTIFACT_LINK / "stage01.sqlite"
if DB_PATH.stat().st_size != expected_db_bytes:
    raise ValueError("DB overlay byte count không khớp artifact manifest.")

# GloVe sidecar tham chiếu các manifest huấn luyện/corpus nằm ngoài thư mục
# mind_large_prepared. Liên kết đúng các file có hash khớp, không bỏ guard.
GLOVE_SIDECAR = ARTIFACT_LINK / "embeddings" / "word_glove_title_abstract_l2.manifest.json"
glove_provenance = json.loads(GLOVE_SIDECAR.read_text(encoding="utf-8"))
provenance_refs = [
    ("training_source_manifest", "training_source_manifest_sha256"),
    ("corpus_manifest", "corpus_manifest_sha256"),
    ("original_matrix_path", "original_matrix_sha256"),
    ("original_matrix_manifest_path", "original_matrix_manifest_sha256"),
]
input_roots = unique_paths(
    [INPUT_DATASET_DIR]
    + [root for root in KAGGLE_DATASETS_ROOT.iterdir() if root.is_dir()]
    + list(prepared_root.parents)
)
GLOVE_PROVENANCE_LINKS = []
for path_key, hash_key in provenance_refs:
    value = glove_provenance.get(path_key)
    expected_hash = glove_provenance.get(hash_key)
    if not value or not expected_hash:
        raise ValueError("GloVe sidecar thiếu provenance field: " + path_key + "/" + hash_key)
    rel = Path(value)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError("GloVe provenance path không relocatable/an toàn: " + str(rel))
    candidates = unique_paths([root / rel for root in input_roots if (root / rel).is_file()])
    matching = []
    observed = []
    for candidate in candidates:
        digest = hashlib.sha256(candidate.read_bytes()).hexdigest()
        observed.append({"path": str(candidate), "sha256": digest})
        if digest == expected_hash:
            matching.append(candidate)
    if not matching:
        raise FileNotFoundError(
            "Không tìm thấy GloVe provenance artifact khớp sidecar: " + path_key
            + "; expected_sha256=" + str(expected_hash)
            + "; candidates=" + json.dumps(observed, ensure_ascii=False)
        )
    source = matching[0].resolve()
    target = PROJECT_ROOT / rel
    if target.is_file():
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected_hash:
            raise RuntimeError("GloVe provenance runtime path đã có nội dung khác: " + str(target))
    elif target.is_symlink():
        if target.resolve() != source:
            raise RuntimeError("GloVe provenance symlink đã trỏ sang nguồn khác: " + str(target))
    elif target.exists():
        raise RuntimeError("GloVe provenance target không phải file/symlink: " + str(target))
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
    GLOVE_PROVENANCE_LINKS.append({"field": path_key, "path": str(target), "sha256": expected_hash})

# Mở thử DB theo đúng chế độ Kaggle input chỉ đọc trước khi khởi tạo runner.
# URI được percent-encode và immutable tránh nhu cầu tạo WAL/SHM cạnh DB.
import sqlite3
resolved_db_path = DB_PATH.resolve(strict=True)
readonly_db_uri = resolved_db_path.as_uri() + "?mode=ro&immutable=1"
try:
    preflight_conn = sqlite3.connect(readonly_db_uri, uri=True, timeout=30)
    preflight_conn.execute("PRAGMA query_only=ON")
    preflight_tables = {row[0] for row in preflight_conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table'")}
    if "vocab" not in preflight_tables:
        raise ValueError("SQLite DB thiếu bảng vocab")
    preflight_vocab_field = preflight_conn.execute(
        "SELECT field FROM vocab LIMIT 1").fetchone()
    preflight_conn.close()
except Exception as exc:
    raise RuntimeError(
        "SQLite read-only preflight failed: path=" + str(resolved_db_path)
        + "; bytes=" + str(resolved_db_path.stat().st_size)
        + "; uri=" + readonly_db_uri + "; error=" + repr(exc)
    ) from exc

print(json.dumps({
    "source_root": str(code_root),
    "source_files_locked": len(source_hashes),
    "source_payload_sha256": payload_hash,
    "prepared_artifact_root": str(prepared_root),
    "db_bytes": DB_PATH.stat().st_size,
    "db_sha256_checked_here": False,
    "sqlite_readonly_preflight": "PASS",
    "sqlite_preflight_vocab_sample": preflight_vocab_field,
    "glove_provenance_files_verified": GLOVE_PROVENANCE_LINKS,
    "checkpoint_files_loaded_in_fresh_mode": False,
    "runtime_project_root": str(PROJECT_ROOT),
}, indent=2))

In [ ]:
# Tạo virtualenv riêng một lần. Không cài TensorFlow vào Python 3.13 của kernel.
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("Không tìm thấy uv trong Kaggle image; không dùng pip của kernel Python 3.13 để thay thế.")
free = shutil.disk_usage(WORK).free
if free < MIN_FREE_DISK_BYTES:
    raise RuntimeError(f"Disk còn {free} bytes, dưới ngưỡng {MIN_FREE_DISK_BYTES}; dừng trước khi tạo môi trường.")

uv_env = dict(os.environ)
uv_env["UV_CACHE_DIR"] = str(WORK / ".uv-cache-mind15")
uv_env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

def run_setup(command):
    result = subprocess.run(command, capture_output=True, text=True, env=uv_env)
    print(result.stdout[-12000:])
    if result.stderr:
        print(result.stderr[-12000:])
    if result.returncode:
        raise RuntimeError("Lệnh môi trường thất bại (rc={}): {}".format(result.returncode, command))

if not TF_PYTHON.is_file():
    run_setup([UV, "python", "install", "3.11.15"])
    run_setup([UV, "venv", "--python", "3.11.15", str(TF_ENV)])
    run_setup([
        UV, "pip", "install", "--python", str(TF_PYTHON),
        "tensorflow[and-cuda]==2.15.1",
        "numpy==1.26.4",
        "scipy==1.17.1",
        "scikit-learn==1.4.2",
    ])
else:
    print("Dùng lại virtualenv đã có; không cài lại package:", TF_ENV)

# GPU 0 là một T4 theo cấu hình Kaggle thông thường; không bật strategy nhiều GPU.
# Phải hoàn thành matmul trên GPU trước khi cho phép training.
probe_path = WORK / "mind15_tf215_gpu_preflight.py"
probe_lines = [
    "import json, tensorflow as tf",
    "gpus = tf.config.list_physical_devices('GPU')",
    "if not gpus: raise SystemExit('TensorFlow 2.15.1 không thấy GPU; training bị chặn.')",
    "for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)",
    "with tf.device('/GPU:0'):",
    "    a = tf.random.uniform([1024, 1024], dtype=tf.float32)",
    "    b = tf.random.uniform([1024, 1024], dtype=tf.float32)",
    "    c = tf.matmul(a, b)",
    "    checksum = float(tf.reduce_sum(c).numpy())",
    "if 'GPU:0' not in c.device.upper(): raise SystemExit('Matmul không chạy trên GPU: ' + c.device)",
    "print(json.dumps({'tensorflow': tf.__version__, 'keras': tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'bundled', 'gpu_count_visible': len(gpus), 'matmul_device': c.device, 'checksum': checksum}))",
]
probe_path.write_text("\n".join(probe_lines) + "\n")
gpu_env = dict(os.environ)
gpu_env["CUDA_VISIBLE_DEVICES"] = "0"
gpu_env["TF_CPP_MIN_LOG_LEVEL"] = "2"
gpu_env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
probe = subprocess.run([str(TF_PYTHON), str(probe_path)], capture_output=True, text=True, env=gpu_env)
print(probe.stdout[-6000:])
if probe.returncode:
    print(probe.stderr[-6000:])
    raise RuntimeError("GPU preflight thất bại; chưa chạy recommendation training.")
print("GPU PREFLIGHT ĐẠT: phép toán TensorFlow thật đã hoàn tất trên GPU 0.")

In [ ]:
# Cấu hình lấy từ JSON canonical đã copy; không thay sampling/model/hyperparameters.
training_config = config_data["training"]
execution_config = config_data["execution"]
effective_batch = int(training_config["effective_batch_size"])
microbatch = int(training_config["microbatch_size"])
seed = int(training_config["partition_base_seed"])

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
if RUN_MODE == "fresh":
    RUN_DIR = RUNS_ROOT / f"partition-{PARTITION}-{stamp}-fresh-u0"
    if RUN_DIR.exists():
        raise FileExistsError("Fresh output directory đã tồn tại; không ghi đè: " + str(RUN_DIR))
    train_args = []
else:
    source_run = Path(RESUME_RUN_DIR).resolve()
    if not (source_run / "run_config.json").is_file() or not (source_run / "checkpoints" / "checkpoint").is_file():
        raise FileNotFoundError("Resume source thiếu run_config/checkpoint: " + str(source_run))
    RUN_DIR = RUNS_ROOT / f"partition-{PARTITION}-{stamp}-resume-copy"
    total_bytes = sum(p.stat().st_size for p in source_run.rglob("*") if p.is_file())
    free = shutil.disk_usage(WORK).free
    if free < total_bytes + MIN_FREE_DISK_BYTES:
        raise RuntimeError("Không đủ disk để copy resume run an toàn; source vẫn được giữ nguyên.")
    shutil.copytree(source_run, RUN_DIR)
    train_args = ["--resume"]

if effective_batch != 1024:
    raise ValueError("Notebook này chỉ chạy effective batch 1024 theo cấu hình khóa.")
if not 1 <= microbatch <= effective_batch:
    raise ValueError("microbatch không hợp lệ theo cấu hình.")
free = shutil.disk_usage(WORK).free
if free < MIN_FREE_DISK_BYTES:
    raise RuntimeError("Disk dưới ngưỡng an toàn trước training; không tự xóa file khác.")

command = [
    str(TF_PYTHON), "-m", "mind_news_metadata.full_pipeline", "train",
    "--db", str(DB_PATH),
    "--partition", str(PARTITION),
    "--seed", str(seed),
    "--effective-batch", str(effective_batch),
    "--microbatch", str(microbatch),
    "--output-dir", str(RUN_DIR),
] + train_args
train_env = dict(gpu_env)
train_env["PYTHONPATH"] = str(PROJECT_ROOT)
train_env["PYTHONUNBUFFERED"] = "1"
LOG_PATH = RUN_DIR.parent / (RUN_DIR.name + ".log")
print(json.dumps({
    "mode": RUN_MODE,
    "partition": PARTITION,
    "effective_batch_size": effective_batch,
    "microbatch_size": microbatch,
    "seed": seed,
    "checkpoint_every_updates": int(execution_config["checkpoint_every_updates"]),
    "output_run_dir": str(RUN_DIR),
    "log_path": str(LOG_PATH),
    "command": command,
    "free_disk_bytes": free,
    "note": "Fresh mode không đọc ckpt-50 từ input; không có max-updates hay giảm mẫu.",
}, indent=2))

In [ ]:
# Chạy một partition đầy đủ theo iterator. Tiến độ lấy từ file checkpoint được ghi
# bởi pipeline; không benchmark và không thay đổi cấu hình giữa các microbatch.
import re

started = time.monotonic()
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(command, cwd=str(PROJECT_ROOT), env=train_env,
                               stdout=log_file, stderr=subprocess.STDOUT)
    last_update = None
    while process.poll() is None:
        time.sleep(60)
        checkpoint_indices = []
        for path in (RUN_DIR / "checkpoints").glob("ckpt-*.index"):
            match = re.search(r"ckpt-(\d+)\.index$", path.name)
            if match:
                checkpoint_indices.append(int(match.group(1)))
        current_update = max(checkpoint_indices) if checkpoint_indices else None
        if current_update != last_update:
            print(json.dumps({
                "status": "TRAINING",
                "latest_checkpoint_update": current_update,
                "elapsed_seconds": round(time.monotonic() - started, 1),
                "free_disk_bytes": shutil.disk_usage(WORK).free,
            }), flush=True)
            last_update = current_update
    return_code = process.returncode

elapsed = time.monotonic() - started
print("runner_return_code:", return_code)
print("elapsed_seconds:", round(elapsed, 1))
if LOG_PATH.is_file():
    tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
    print("\n".join(tail))
if return_code != 0:
    raise RuntimeError("Training runner lỗi. Giữ output/checkpoint hiện có; không tự restart từ update 0.")
progress_path = RUN_DIR / "training_progress.json"
if not progress_path.is_file():
    raise RuntimeError("Runner kết thúc nhưng thiếu training_progress.json; không tuyên bố hoàn tất.")
progress = json.loads(progress_path.read_text(encoding="utf-8"))
print(json.dumps(progress, indent=2))
if not progress.get("full_epoch", False):
    print("LƯU Ý: output không được đánh dấu full_epoch; không gọi đây là một epoch hoàn tất.")

## Sau khi training

- Xác nhận `training_progress.json` có `full_epoch: true` và `status: EPOCH_ITERATOR_EXHAUSTED` trước khi gọi partition 0 hoàn tất.
- `checkpoints/` giữ tối đa hai checkpoint gần nhất. Log ở cạnh thư mục run. Giữ notebook/dataset ở chế độ Private.
- Muốn đánh giá dev, chạy cell tùy chọn bên dưới sau khi training kết thúc. Đây là dev dùng chọn cấu hình, không phải holdout độc lập.
- Muốn tiếp tục trong phiên sau, lưu output run làm input Private rồi đặt `RUN_MODE="resume"`, `RESUME_RUN_DIR` trỏ chính xác tới run đó. Notebook sao chép run sang `/kaggle/working`; pipeline phải chấp nhận fingerprint chính xác. Nếu bị từ chối, không ép resume và không chuyển sang checkpoint khác.

In [ ]:
# Tùy chọn: full-dev evaluation sau khi đã train xong. Mặc định không chạy.
RUN_DEV_EVALUATION = False
# Required only for report-based dev masking; build this from raw train behaviors.tsv.
TRAIN_DID_MEMBERSHIP_PATH = ""
if RUN_DEV_EVALUATION:
    if not TRAIN_DID_MEMBERSHIP_PATH:
        raise FileNotFoundError("dev_new_did_masked requires a membership .npy built from raw train behaviors.tsv")
    eval_command = [
        str(TF_PYTHON), "-m", "mind_news_metadata.full_pipeline", "evaluate",
        "--run-dir", str(RUN_DIR), "--db", str(DB_PATH),
        "--split", "dev", "--profile", "dev_new_did_masked",
        "--train-dids", str(Path(TRAIN_DID_MEMBERSHIP_PATH).resolve()),
        "--batch-size", str(int(execution_config["evaluation_batch_size"])),
    ]
    eval_env = dict(gpu_env)
    eval_env["PYTHONPATH"] = str(PROJECT_ROOT)
    subprocess.run(eval_command, cwd=str(PROJECT_ROOT), env=eval_env, check=True)
else:
    print("Dev evaluation tidak dijalankan. Bật RUN_DEV_EVALUATION hanya setelah run train berhasil.")